# 04 · Monitor independente, anotação humana e inferência (v4)

**Ambiente: GPU.** Aplica o monitor independente congelado às 120 avaliações GRPO, exporta a
amostra cega de 360 itens e calcula os contrastes por semente (inversão de sinais, bootstrap,
Holm). Os arquivos humanos devem ser preenchidos por pessoas e salvos em `human/`:
`completed_annotations.csv` (annotation_id, rater, label), `adjudication.csv` (annotation_id, label)
e `independent_monitor_controls.csv` (question, trace, label, source=human, split=heldout), com
pelo menos 100 positivos e 100 negativos. Sem eles, o relatório permanece `INSUFFICIENT_EVIDENCE`.


In [ ]:
# Configuração operacional. Sementes, tamanhos, modelos e critérios estão fixos no protocolo v4.
STUDY_ID = "estudo-prospectivo-2026-v4"  # @param {type:"string"}
STORAGE_MODE = "auto"  # @param ["auto", "local"]
TECHNICAL_TEST = False  # @param {type:"boolean"}
MAX_JOBS_PER_SESSION = 4  # @param {type:"integer"}
CODE_COMMIT = "b1426de6365744b37fea4d857c77200b3de1fbf0"
CODE_HASHES = {
 "__init__.py": "903b3bdb88f326693084e14633a5ebce1f862506124b58f67dc751c2a2c96f76",
 "avaliacao.py": "01b6f0f4f9e651dd2a1904eb19da2a8066a9366b976b768911722b9ada8d966f",
 "contrato.py": "d5829c2bbeb14e7bff84a6ffe757642145aad4ce57939c11c5ffc23b2b3bb43b",
 "estatistica.py": "7fd7615c0ead1e01c0d7e926e41c56a0b203b4ba493f3153362a9779c945bdfd",
 "modelos.py": "ee02866b8eac6db6928233805f50c985352d5c57253a3bfcf01d7c56fe910811",
 "monitor.py": "3635ed93e186e64dea87b0df428c8472434f40e2d243660b7ffb63ddce2a87f3",
 "runtime.py": "c1f4326e798b39e8b90fc6099d1c8da8b31e14a0ec9b46bf678c43c2a77fe3b0",
 "tarefas.py": "8b1a58ecc9df6b959741409f71811b7e90449bc41eedf7d860528f74732ba7fd",
 "treino.py": "2f672f04a987090d268b1b3e2da5291d2d19bb6ef5fc772aba83a4c9f40d0975"
}
SUPPORT_HASHES = {
 "armazenamento.py": "b26088c7e2339b0c9968cad6f943db05a4effe27aa9eb5262bf84830e361f1f0",
 "sessao.py": "3c8c3774d66fce7fc0881bfb2db0e65d7bfbb2a2a38ef36f48373ddba29dde41"
}

import hashlib, importlib.util, os, re, subprocess, sys, time
from pathlib import Path
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", STUDY_ID):
    raise ValueError("STUDY_ID aceita somente letras, números, _ e -.")
if TECHNICAL_TEST and not STUDY_ID.endswith("-teste-tecnico"):
    STUDY_ID += "-teste-tecnico"  # pasta separada: nunca se mistura ao estudo
try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    drive, ON_COLAB = None, False
LOCAL = Path(os.environ.get("COT_LOCAL_ROOT", "/content/cot-v4" if ON_COLAB else Path.cwd() / "cot-v4"))
LOCAL.mkdir(parents=True, exist_ok=True)

def checkout(ref):
    """Obtém exatamente o commit fixado; não usa clones antigos nem a branch atual."""
    override = os.environ.get("COT_REPO_DIR")
    if override:
        return Path(override).resolve()
    repo = LOCAL / "source" / ref
    git = lambda *a, **k: subprocess.run(["git", "-C", str(repo), *a], capture_output=True, text=True, **k)
    if not (repo / ".git").is_dir():
        repo.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "--quiet", str(repo)], check=True)
        git("remote", "add", "origin", "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git")
    if git("cat-file", "-e", ref + "^{commit}").returncode:
        for attempt in range(1, 4):
            fetched = git("fetch", "--quiet", "--depth", "1", "origin", ref)
            if not fetched.returncode:
                break
            print(f"Falha de rede ao baixar o código (tentativa {attempt}/3): {fetched.stderr.strip()[-300:]}")
            time.sleep(5 * attempt)
        else:
            raise RuntimeError("Não foi possível baixar o commit " + ref + ". Verifique a conexão e execute de novo.")
    if git("checkout", "--quiet", "--detach", ref).returncode:
        raise RuntimeError("Checkout do commit fixado falhou: " + ref)
    if git("rev-parse", "HEAD").stdout.strip() != ref:
        raise RuntimeError("O código aberto não corresponde ao commit do notebook.")
    return repo

REPO = checkout(CODE_COMMIT)
PACKAGE = REPO / "experimento_cientifico_v4"
def load(name):
    path = PACKAGE / "colab" / name
    if hashlib.sha256(path.read_bytes()).hexdigest() != SUPPORT_HASHES[name]:
        raise RuntimeError("Suporte divergente do notebook: " + name + ". Reabra pelo link do GitHub.")
    # Nome único por commit: módulos de versões antigas na mesma sessão não são reutilizados.
    spec = importlib.util.spec_from_file_location(f"cotv4_{name[:-3]}_{CODE_COMMIT[:12]}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
armazenamento, sessao = load("armazenamento.py"), load("sessao.py")
STATE = armazenamento.prepare_storage(LOCAL, STUDY_ID, on_colab=ON_COLAB, mode=STORAGE_MODE,
    storage_override=os.environ.get("COT_STORAGE_ROOT"), drive_module=drive,
    mountpoint=os.environ.get("COT_DRIVE_MOUNTPOINT", "/content/drive"))
STUDY = sessao.Session(REPO, STATE, CODE_COMMIT, CODE_HASHES, on_colab=ON_COLAB, technical_test=TECHNICAL_TEST)
Blocked = sessao.StageBlocked
print("Código:", CODE_COMMIT, "| Estudo:", STUDY_ID, "| Teste técnico:", TECHNICAL_TEST)


In [ ]:
status = STUDY.execute("status")
if not status["all_evaluation_complete"]:
    print("ETAPA BLOQUEADA: conclua todas as avaliações no notebook 03:", status["evaluation"])
elif "grpo" in status.get("excluded", {}):
    print("NÃO SE APLICA: braço GRPO excluído deste estudo antes do registro:", status["excluded"]["grpo"])
else:
    try:
        print(STUDY.execute("monitor", mode="grpo", max_jobs=MAX_JOBS_PER_SESSION))
        print(STUDY.execute("blind", mode="grpo"))
        if (STUDY.storage / "human" / "independent_monitor_controls.csv").is_file():
            print(STUDY.execute("controls", mode="grpo"))
        else:
            print("PENDENTE: human/independent_monitor_controls.csv com rótulos humanos.")
    except Blocked as reason:
        print("ETAPA BLOQUEADA:", reason)


In [ ]:
report = STUDY.execute("report")
print("Veredito:", report["verdict"])
print("Pendências:", report["blocked_by"])
STUDY.export()
STUDY.status()


**Situação das etapas.** CONCLUÍDA: arquivos verificados por hash. BLOQUEADA: um critério do
protocolo ou um recurso impede a etapa (o motivo é impresso). NÃO EXECUTADA / PENDENTE: ainda
não iniciada. Reexecutar retoma o mesmo `STUDY_ID`; trocar o ID inicia outro estudo.
Cópias antigas salvas no Colab mantêm o commit antigo: abra sempre pelo link do README.
